# Algoritmos de machine learning: ejercicios

Un ejercicio breve por algoritmo, con datos de juguete y semilla fija para que el resultado se pueda repetir.

Explicación, fórmulas y simuladores en el artículo: https://stivenson.github.io/#/articles/algoritmos-ml-explorador

Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**. No hace falta GPU.

## Linear Regression (regresión lineal)

Precio de casas: ¿cuánto suma cada m² y cada habitación?

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=linear-regression&tab=realWorld)

In [ ]:
# Precio de casas: ¿cuánto suma cada m² y cada habitación?
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

rng = np.random.default_rng(42)
n = 200
area = rng.uniform(40, 200, n)          # m²
habitaciones = rng.integers(1, 6, n)    # de 1 a 5
# Precio "real" en millones: base + 2.5 por m² + 15 por habitación + ruido
precio = 80 + 2.5 * area + 15 * habitaciones + rng.normal(0, 25, n)

X = np.column_stack([area, habitaciones])
X_train, X_test, y_train, y_test = train_test_split(X, precio, test_size=0.25, random_state=0)

modelo = LinearRegression().fit(X_train, y_train)
pred = modelo.predict(X_test)

print(f"Intercepto b0: {modelo.intercept_:.1f}")
print(f"Por cada m² (b1): {modelo.coef_[0]:.2f}")
print(f"Por cada habitación (b2): {modelo.coef_[1]:.1f}")
print(f"R² con casas que no vio: {r2_score(y_test, pred):.3f}")
print(f"Error medio: {mean_absolute_error(y_test, pred):.1f} millones")
print(f"Casa de 120 m² y 3 habitaciones: {modelo.predict([[120, 3]])[0]:.0f} millones")

plt.figure(figsize=(4.5, 3.2))
plt.scatter(y_test, pred, s=14)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("Precio real")
plt.ylabel("Precio predicho")
plt.title("Más cerca de la diagonal = mejor")
plt.show()

## Logistic Regression (regresión logística)

Detector de spam: probabilidad de que un correo sea spam según tres señales.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=logistic-regression&tab=realWorld)

In [ ]:
# Detector de spam: probabilidad de que un correo sea spam según 3 señales
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

rng = np.random.default_rng(7)
n = 400
es_spam = rng.random(n) < 0.4
gratis = np.where(es_spam, rng.poisson(3, n), rng.poisson(0.3, n))     # veces que dice "gratis"
enlaces = np.where(es_spam, rng.poisson(5, n), rng.poisson(1.5, n))    # número de enlaces
conocido = np.where(es_spam, rng.random(n) < 0.1, rng.random(n) < 0.7).astype(int)  # remitente conocido

X = np.column_stack([gratis, enlaces, conocido])
y = es_spam.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

modelo = LogisticRegression().fit(X_train, y_train)
pred = modelo.predict(X_test)

print("Pesos [gratis, enlaces, conocido]:", np.round(modelo.coef_[0], 2))
print(f"Intercepto b0: {modelo.intercept_[0]:.2f}")
print(f"Exactitud con correos nuevos: {accuracy_score(y_test, pred):.1%}")
print("Matriz de confusión (filas = real, columnas = predicho; 0 = normal, 1 = spam):")
print(confusion_matrix(y_test, pred))

correos = np.array([[0, 1, 1], [2, 4, 0], [5, 8, 0]])
for c, p in zip(correos, modelo.predict_proba(correos)[:, 1]):
    print(f"gratis={c[0]} enlaces={c[1]} conocido={c[2]} -> P(spam) = {p:.1%}")

## Decision Tree (árbol de decisión)

¿Pagará el préstamo? Un árbol que se lee como reglas, y qué le pasa cuando crece demasiado.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=decision-tree&tab=realWorld)

In [ ]:
# ¿Pagará el préstamo? Un árbol que se lee como reglas "si... entonces..."
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(3)
n = 500
ingreso = rng.uniform(1, 12, n).round(1)   # millones al mes
deuda = rng.uniform(0, 0.9, n).round(2)    # fracción del ingreso ya comprometida
atrasos = rng.poisson(0.8, n)              # pagos atrasados el último año
riesgo = 0.6 * deuda + 0.15 * atrasos - 0.04 * ingreso + rng.normal(0, 0.08, n)
impago = (riesgo > 0.25).astype(int)
# 12 % de etiquetas al azar: en la vida real hay casos que ninguna regla explica
ruido = rng.random(n) < 0.12
impago[ruido] = 1 - impago[ruido]

X = np.column_stack([ingreso, deuda, atrasos])
X_train, X_test, y_train, y_test = train_test_split(X, impago, test_size=0.3, random_state=0)

arbol = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(arbol, feature_names=["ingreso", "deuda", "atrasos"], class_names=["paga", "impago"]))

print("Profundidad | exactitud entrenamiento | exactitud datos nuevos")
for prof in [1, 3, 5, 15]:
    a = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(X_train, y_train)
    print(f"{prof:>11} | {a.score(X_train, y_train):>23.1%} | {a.score(X_test, y_test):>22.1%}")

## Random Forest (bosque aleatorio)

Detector de fraude: un árbol solo contra un bosque de 100, y qué señales pesan de verdad.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=random-forest&tab=realWorld)

In [ ]:
# Detector de fraude: un árbol solo contra un bosque de 100 árboles, y qué señales pesan más
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(7)
n = 1000
monto = rng.exponential(80, n)            # monto de la compra (miles de pesos)
hora = rng.integers(0, 24, n)             # hora del día (0 a 23)
distancia = rng.exponential(10, n)        # km desde la ciudad habitual del cliente
intentos = rng.poisson(0.4, n)            # intentos fallidos de clave antes de la compra
antiguedad = rng.integers(1, 120, n)      # meses de la tarjeta: NO influye en el fraude
# Regla oculta: el fraude sube con monto alto, madrugada, lejos de casa e intentos fallidos
riesgo = 0.03 * monto + 2.5 * (hora < 6) + 0.12 * distancia + 1.5 * intentos - 6
fraude = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)

X = np.column_stack([monto, hora, distancia, intentos, antiguedad])
nombres = ["monto", "hora", "distancia", "intentos", "antigüedad"]
X_tr, X_te, y_tr, y_te = train_test_split(X, fraude, test_size=0.3, random_state=0, stratify=fraude)

arbol = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
bosque = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
print(f"Fraudes: {fraude.sum()} de {n} compras ({fraude.mean():.0%})")
print(f"Un árbol solo   → entrenamiento {arbol.score(X_tr, y_tr):.1%} | datos nuevos {arbol.score(X_te, y_te):.1%}")
print(f"Bosque de 100   → entrenamiento {bosque.score(X_tr, y_tr):.1%} | datos nuevos {bosque.score(X_te, y_te):.1%}")

perm = permutation_importance(bosque, X_te, y_te, n_repeats=5, random_state=0)
print("\nImportancia de cada señal      impureza  permutación")
for j in np.argsort(-perm.importances_mean, kind="stable"):
    print(f"  {nombres[j]:<11}                  {bosque.feature_importances_[j]:.2f}       {perm.importances_mean[j]:.3f}")

## Gradient Boosting (potenciación por gradiente)

Scoring de crédito: cada árbol corrige al anterior. La curva de pérdida dice cuándo parar.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=gradient-boosting&tab=realWorld)

In [ ]:
# Scoring de crédito: cada árbol nuevo corrige los errores de los anteriores. ¿Cuándo parar?
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(3)
n = 800
ingreso = rng.normal(4, 1.5, n).clip(0.8)          # millones de pesos al mes
deuda = rng.uniform(0, 0.9, n)                      # deuda / ingreso
atrasos = rng.poisson(0.8, n)                       # pagos atrasados en el último año
edad = rng.integers(20, 70, n)
# Regla oculta (no lineal): la deuda pesa mucho más cuando el ingreso es bajo
riesgo = 3 * deuda * (ingreso < 3) + 1.2 * deuda + 0.7 * atrasos - 0.3 * ingreso + 0.6 * (edad < 25) - 1
impago = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)

X = np.column_stack([ingreso, deuda, atrasos, edad])
X_tr, X_te, y_tr, y_te = train_test_split(X, impago, test_size=0.3, random_state=0, stratify=impago)

# 300 árboles a propósito (el valor por defecto es 100), para ver qué pasa cuando sobran.
# min_samples_leaf=20: cada hoja debe tener al menos 20 clientes.
gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=3, min_samples_leaf=20, random_state=0
)
gb.fit(X_tr, y_tr)

perdida_tr = [log_loss(y_tr, p[:, 1]) for p in gb.staged_predict_proba(X_tr)]
perdida_te = [log_loss(y_te, p[:, 1]) for p in gb.staged_predict_proba(X_te)]
mejor = int(np.argmin(perdida_te)) + 1

print(f"Impagos: {impago.sum()} de {n} clientes")
print("Árboles | pérdida entrenamiento | pérdida datos nuevos")
for m in [1, 10, 50, 100, 200, 300]:
    print(f"{m:>7} | {perdida_tr[m - 1]:>21.3f} | {perdida_te[m - 1]:>20.3f}")
print(f"\nMenor pérdida en datos nuevos: {perdida_te[mejor - 1]:.3f} con {mejor} árboles")

plt.figure(figsize=(4.5, 3.2))
plt.plot(range(1, 301), perdida_tr, label="entrenamiento")
plt.plot(range(1, 301), perdida_te, label="datos nuevos")
plt.axvline(mejor, ls="--", c="gray")
plt.xlabel("número de árboles")
plt.ylabel("pérdida logarítmica")
plt.title("Después de la línea gris, más árboles empeoran")
plt.legend()
plt.show()

## SVM (máquina de vectores de soporte)

Dígitos escritos a mano (8×8 píxeles): kernel lineal contra RBF y el efecto de C.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=svm&tab=realWorld)

In [ ]:
# Reconocer dígitos escritos a mano (imágenes de 8×8 píxeles): SVM lineal contra kernel RBF
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

digitos = load_digits()                  # 1797 imágenes, viene incluido en scikit-learn
X = digitos.data / 16                    # 64 píxeles por imagen, escalados a [0, 1]
y = digitos.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(X_tr)} imágenes para entrenar, {len(X_te)} nuevas para probar")

print("\nKernel  C      acierto en nuevas  vectores de soporte")
for kernel in ["linear", "rbf"]:
    for C in [0.01, 1, 100]:
        svm = SVC(kernel=kernel, C=C).fit(X_tr, y_tr)
        print(f"{kernel:<7} {C:<6} {svm.score(X_te, y_te):>17.1%}  {svm.n_support_.sum():>19}")

svm = SVC(kernel="rbf", C=1).fit(X_tr, y_tr)
errores = np.flatnonzero(svm.predict(X_te) != y_te)
print(f"\nCon RBF y C = 1 falla {len(errores)} de {len(y_te)}. Por ejemplo:")
for i in errores[:3]:
    print(f"  era un {y_te[i]}, dijo {svm.predict(X_te[i:i + 1])[0]}")

fig, ejes = plt.subplots(1, 3, figsize=(4.5, 1.8))
for eje, i in zip(ejes, errores[:3]):
    eje.imshow(X_te[i].reshape(8, 8), cmap="gray")
    eje.set_title(f"{y_te[i]} → {svm.predict(X_te[i:i + 1])[0]}")
    eje.axis("off")
plt.show()

## KNN (k vecinos más cercanos)

Recomendador: qué película ver según lo que les gustó a los usuarios más parecidos.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=knn&tab=realWorld)

In [ ]:
# Recomendador: ¿qué película ver? Lo que les gustó a los usuarios más parecidos a Ana
import numpy as np
from sklearn.neighbors import NearestNeighbors

peliculas = ["Toy Story", "Alien", "Titanic", "Matrix", "Coco", "Interestelar"]
# Calificaciones de 1 a 5 (0 = no la ha visto)
usuarios = {
    "Ana":  [5, 0, 4, 0, 5, 0],
    "Beto": [5, 1, 4, 2, 5, 2],
    "Caro": [1, 5, 1, 5, 1, 5],
    "Dani": [4, 1, 5, 1, 4, 4],
    "Eva":  [2, 5, 1, 5, 2, 4],
    "Fer":  [5, 2, 5, 1, 4, 5],
}
nombres = list(usuarios)
R = np.array(list(usuarios.values()), dtype=float)
vistas = R[0] > 0  # comparamos solo con las películas que Ana ya calificó

knn = NearestNeighbors(n_neighbors=3, metric="euclidean").fit(R[1:][:, vistas])
dist, idx = knn.kneighbors(R[0:1, vistas])
vecinos = idx[0] + 1  # +1 porque Ana (fila 0) quedó fuera del ajuste

for i, d in zip(vecinos, dist[0]):
    print(f"Vecino: {nombres[i]:<5} distancia = {d:.2f}")

# Aquí los 3 vecinos vieron las 6 películas; si alguno tuviera un 0, contaría como nota mínima y habría que excluirlo.
puntaje = R[vecinos].mean(axis=0)
print("\nRecomendaciones para Ana (promedio de sus 3 vecinos):")
for j in np.argsort(-puntaje, kind="stable"):  # estable: los empates salen en orden fijo
    if not vistas[j]:
        print(f"  {peliculas[j]:<13} {puntaje[j]:.1f} / 5")

## Naive Bayes (Bayes ingenuo)

Sentimiento de reseñas: cada palabra multiplica la evidencia a favor de positiva o negativa.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=naive-bayes&tab=realWorld)

In [ ]:
# Sentimiento de reseñas: Naive Bayes cuenta palabras y multiplica sus evidencias
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

resenas = [
    ("excelente producto llegó rápido y funciona perfecto", 1),
    ("muy buena calidad lo recomiendo", 1),
    ("me encantó excelente atención", 1),
    ("funciona perfecto buena compra", 1),
    ("rápido y buena calidad recomendado", 1),
    ("excelente precio muy contento", 1),
    ("buena batería y pantalla excelente", 1),
    ("lo recomiendo a todos me encantó", 1),
    ("pésimo producto llegó roto", 0),
    ("muy mala calidad no lo recomiendo", 0),
    ("llegó tarde y no funciona", 0),
    ("mala atención pésimo servicio", 0),
    ("se dañó en una semana mala compra", 0),
    ("no funciona devolví el producto", 0),
    ("la batería dura poco mala calidad", 0),
    ("pésimo no lo compren", 0),
]
textos = [t for t, _ in resenas]
y = np.array([s for _, s in resenas])

vectorizador = CountVectorizer()
X = vectorizador.fit_transform(textos)
nb = MultinomialNB(alpha=1.0).fit(X, y)   # alpha = 1: suavizado de Laplace
print(f"{len(textos)} reseñas, {X.shape[1]} palabras distintas")
print(f"Prior: P(positiva) = {np.exp(nb.class_log_prior_[1]):.2f}")

nuevas = ["excelente calidad llegó rápido", "no funciona mala compra", "llegó la batería nueva"]
print("\nReseña nueva                     P(positiva)")
for texto, p in zip(nuevas, nb.predict_proba(vectorizador.transform(nuevas))[:, 1]):
    print(f"  {texto:<31} {p:.0%}")

palabras = vectorizador.get_feature_names_out()
razon = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]   # log P(w|pos) - log P(w|neg)
print("\nPalabras que más empujan a positiva:", ", ".join(palabras[np.argsort(-razon, kind="stable")[:3]]))
print("Palabras que más empujan a negativa:", ", ".join(palabras[np.argsort(razon, kind="stable")[:3]]))
for w in ["llegó", "batería", "la"]:
    print(f"«{w}» multiplica los odds de positiva por {np.exp(razon[vectorizador.vocabulary_[w]]):.2f}")
print("«nueva» no estaba en el entrenamiento: se ignora")

## DBSCAN (agrupamiento por densidad)

Reportes de huecos en un mapa: zonas de forma libre y puntos aislados como ruido.

[Abrir en el explorador](https://stivenson.github.io/#/articles/algoritmos-ml-explorador?alg=dbscan&tab=realWorld)

In [ ]:
# Clustering geoespacial: ¿dónde se concentran los reportes de huecos en la vía?
import numpy as np
from sklearn.cluster import DBSCAN, KMeans

rng = np.random.default_rng(5)
R = 6371.0  # radio de la Tierra en km
lat0, lon0 = 10.98, -74.80  # un punto de referencia en la ciudad
km = lambda n, s: rng.normal(0, s, size=(n, 2))  # n puntos dispersos s km

# Coordenadas en km respecto al punto de referencia:
barrio = km(40, 0.15) + [1.0, 1.0]  # zona compacta
t = rng.uniform(0, 3, size=60)  # avenida: franja larga y angosta
avenida = np.c_[t - 2.0, 0.6 * t - 1.5] + km(60, 0.05)
mercado = km(30, 0.12) + [-1.5, 1.8]
sueltos = rng.uniform(-3, 3, size=(15, 2))  # reportes aislados
xy = np.vstack([barrio, avenida, mercado, sueltos])
grados = np.c_[lat0 + np.degrees(xy[:, 1] / R), lon0 + np.degrees(xy[:, 0] / (R * np.cos(np.radians(lat0))))]
print(f"{len(grados)} reportes (latitud, longitud)")

# La distancia haversine trabaja en radianes sobre la esfera: eps en metros ÷ 1000 ÷ R.
for metros in (150, 300, 800):
    db = DBSCAN(eps=metros / 1000 / R, min_samples=5, metric="haversine").fit(np.radians(grados))
    zonas = sorted(np.bincount(db.labels_[db.labels_ >= 0]).tolist(), reverse=True)
    print(f"eps = {metros:3d} m → {len(zonas)} zonas {zonas}, ruido: {np.sum(db.labels_ == -1)}")

km3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(xy)  # K-Means en km
en_avenida = km3.labels_[40:100]
partidos = np.sum(en_avenida != np.bincount(en_avenida).argmax())
lejos = np.linalg.norm(sueltos - km3.cluster_centers_[km3.labels_[-15:]], axis=1)
print(f"\nK-Means (K = 3) manda {partidos} de los 60 reportes de la avenida a otra zona")
print(f"y mete los 15 aislados en alguna zona: el más lejano queda a {lejos.max():.1f} km de su centro")